# Métricas de evaluación para textos generados por modelos de lenguaje

**Curso:** Procesamiento de Lenguaje Natural · Unidad 5 — Clasificación de texto y evaluación  
**Institución:** Universidad Tecnológica de Bolívar  
**Referencia:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3.ª ed.), Cap. 3 — N-gram Language Models (perplejidad) y Cap. 13 — Machine Translation (BLEU)  
**Compatibilidad con Python:** 3.10 · 3.11 · 3.12 · 3.13

## Contexto y Propósito

Este notebook muestra cómo **medir la calidad de un texto generado** por un modelo de lenguaje. Genera un resumen con **T5** (arquitectura codificador–decodificador) y una continuación con **GPT-2** (solo decodificador), y los evalúa con dos familias de métricas: las que comparan el texto generado con una referencia escrita por una persona (**BLEU** y **ROUGE**) y la que mide qué tan “sorprendido” queda el modelo ante un texto (**perplejidad**). Termina con una tabla de qué mide cada métrica y sus límites.

**Objetivo:** saber elegir e interpretar métricas automáticas para tareas de generación (resumen, traducción, respuesta), entendiendo que ninguna mide por sí sola si un texto es correcto o útil.

## Ejemplos de Aplicación en el Mundo Real

Cada caso muestra, paso a paso, cómo se aplica la técnica de este notebook a un problema real.

### Finanzas y servicio al cliente — Resúmenes de llamadas

- **Problema:** decidir si el resumen automático puede reemplazar al que escribe el supervisor.
- **Entrada:** transcripción de la llamada y resumen del supervisor.
- **Cómo se aplica la evaluación de texto generado:**
  1. T5 genera el resumen de cada llamada.
  2. ROUGE mide el solapamiento con el resumen del supervisor.
  3. Una muestra se revisa a mano, porque ROUGE no detecta datos inventados.
- **Salida:** ROUGE-1 y ROUGE-L por llamada.
- **Decisión:** piloto solo si ROUGE y la revisión humana superan el umbral acordado.

### Energía e industria — Reportes anómalos con perplejidad

- **Problema:** detectar reportes de turno mal diligenciados o inusuales.
- **Entrada:** reportes de turno.
- **Cómo se aplica la perplejidad:**
  1. Un modelo de lenguaje (GPT-2 o uno ajustado a los reportes) calcula la perplejidad de cada reporte.
  2. Una perplejidad alta indica que el texto es inusual para el modelo.
  3. Se ordenan los reportes de mayor a menor perplejidad.
- **Salida:** ranking de reportes inusuales.
- **Decisión:** el supervisor revisa los primeros del ranking.

## Requisitos Previos

**Instalación** (una sola vez, desde la terminal; en Jupyter o Colab antepón `!`):

```bash
pip install torch transformers evaluate sacrebleu rouge_score sentencepiece
```

**Librerías que se importan:**

| Librería | Para qué se usa |
|---------|-----------------|
| `transformers` | `AutoTokenizer`, `AutoModelForSeq2SeqLM` (T5) y `AutoModelForCausalLM` (GPT-2) |
| `evaluate` | Implementaciones estándar de BLEU (`sacrebleu`) y ROUGE |
| `math, torch` | Cálculo de la perplejidad |

**Modelos:** `t5-small` (~240 MB) y `gpt2` (~550 MB) se descargan de Hugging Face la primera vez. Las métricas de `evaluate` también se descargan del Hub.

## Marco metodológico: pipeline ML / LLM (4×3)

![Pipeline ML y LLM](img/Pipeline_ML_LLM_4x3.png)

El diagrama tiene una franja común de datos (arriba), dos rutas de modelado y una franja de operación (abajo): **ruta clásica** (*feature-based ML*) y **ruta LLM** (*foundation models*). Este notebook recorre la **ruta LLM / fundacional**. Etapas del diagrama que cubre:

| Etapa del diagrama | Dónde se aborda en este notebook |
|--------------------|----------------------------------|
| Foundation / Pre-trained LLM | Sección 1: T5 y GPT-2 preentrenados |
| Prompt Engineering / In-Context Learning | Sección 3: el prefijo de tarea `summarize:` de T5 |
| LLM Outputs | Secciones 3–4: resumen de T5 y continuación de GPT-2 |
| Test / Eval Set | Sección 2: texto de entrada y resumen de referencia |
| Evaluation & QA | Secciones 5–6: BLEU, ROUGE y perplejidad |

El foco está en la caja **Evaluation & QA** de la ruta LLM. Nota: el nombre del archivo (`classification_evaluation`) no corresponde a su contenido, que trata de evaluación de **generación**; para métricas de clasificación ver `bert_classifier.ipynb` o `metrics_llm_business.ipynb`.

## Estándares de referencia

Prácticas de este notebook alineadas con estándares de IEEE SA, ISO/IEC y la Open Source Initiative (OSI). Se citan como guía de buenas prácticas, no como certificación de cumplimiento.

| Estándar | Cómo se aplica aquí |
|----------|---------------------|
| **IEEE 3119-2025** — Adquisición (*procurement*) de IA y sistemas de decisión automatizada | Evaluar un modelo generativo de terceros con métricas explícitas, y conocer sus límites, es parte de una adquisición responsable de IA. |
| **IEEE 7001-2021** — Transparencia de sistemas autónomos | La tabla final documenta qué mide cada métrica y qué no, para que nadie interprete un ROUGE alto como “texto correcto”. |
| **ISO/IEC 23894:2023** — Guía para la gestión de riesgos de IA | Riesgo documentado: las métricas de solapamiento no detectan errores factuales ni alucinaciones; se necesita revisión humana o métricas semánticas. |
| **OSI · Open Source AI Definition 1.0 (2024)** — Qué hace que un sistema de IA sea de código abierto: libertades de usar, estudiar, modificar y compartir, con acceso a código, parámetros e información de los datos | T5 y GPT-2 publican pesos y código, y `evaluate` es una librería abierta: los resultados se pueden reproducir. |

## Objetivos de aprendizaje

Al finalizar este notebook podrás:

1. Generar texto con **T5** (codificador–decodificador) y **GPT-2** (solo decodificador)
2. Evaluar la calidad de la generación con las métricas **BLEU** y **ROUGE**
3. Calcular la **perplejidad** como medida intrínseca de la calidad de un modelo de lenguaje
4. Comparar las fortalezas y limitaciones de cada métrica

---

## 1. Cargar los modelos

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
)
import evaluate
import math
import torch

torch.manual_seed(42)   # GPT-2 genera con muestreo: la semilla hace reproducible el ejemplo

T5_MODEL_NAME = 't5-small'
GPT2_MODEL_NAME = 'gpt2'

# T5: codificador–decodificador, preentrenado con tareas texto→texto (resumen, traducción…)
t5_tokenizer = AutoTokenizer.from_pretrained(T5_MODEL_NAME)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL_NAME)

# GPT-2: solo decodificador, modelo de lenguaje generativo
gpt2_tokenizer = AutoTokenizer.from_pretrained(GPT2_MODEL_NAME)
gpt2_model = AutoModelForCausalLM.from_pretrained(GPT2_MODEL_NAME)
gpt2_model.eval()

print(f'T5 cargado: {T5_MODEL_NAME}')
print(f'GPT-2 cargado: {GPT2_MODEL_NAME}')

> **Interpretación de la salida:** ambos modelos se descargan del Hub de Hugging Face la primera vez y quedan en caché local. T5-small tiene ~60 millones de parámetros y se preentrenó con tareas como resumen y traducción. GPT-2 (versión base) tiene ~124 millones de parámetros y está pensado para generar texto palabra a palabra.

---

## 2. Texto de entrada y texto de referencia

El texto de referencia es el resumen “ideal” escrito por una persona, contra el cual se comparan las salidas.

In [ ]:
# Texto a resumir y resumen de referencia (escrito por una persona)
input_text = 'The Eiffel Tower is located in Paris and was built in 1889.'
reference_summary = 'The Eiffel Tower was built in Paris in 1889.'

---

## 3. Generar texto con T5

In [ ]:
# T5 necesita un prefijo de tarea; 'summarize:' le indica que resuma
t5_input = 'summarize: ' + input_text
inputs_t5 = t5_tokenizer(t5_input, return_tensors='pt')
# Decodificación voraz (sin muestreo): el resultado es determinista
output_t5 = t5_model.generate(**inputs_t5, max_length=30)
text_t5 = t5_tokenizer.decode(output_t5[0], skip_special_tokens=True)
print(f'Salida de T5: {text_t5}')

---

## 4. Generar texto con GPT-2

In [ ]:
inputs_gpt2 = gpt2_tokenizer(input_text, return_tensors='pt')
# Muestreo con temperatura 0.7 entre los 50 tokens más probables
output_gpt2 = gpt2_model.generate(
    **inputs_gpt2,
    max_length=30,
    temperature=0.7,
    do_sample=True,
    top_k=50,
    pad_token_id=gpt2_tokenizer.eos_token_id,   # GPT-2 no tiene token de relleno
)
text_gpt2 = gpt2_tokenizer.decode(output_gpt2[0], skip_special_tokens=True)
print(f'Salida de GPT-2: {text_gpt2}')

> **Interpretación de la salida:** T5 se entrenó con tareas de resumen, así que produce una paráfrasis comprimida del texto. GPT-2 es un modelo de lenguaje general: sin ajuste para la tarea, **continúa** la oración en lugar de resumirla. Esta diferencia muestra la importancia del ajuste a instrucciones (*instruction tuning*) para la generación orientada a tareas.

---

## 5. Evaluación con BLEU y ROUGE

In [ ]:
# Métricas estándar de la librería evaluate
bleu = evaluate.load('sacrebleu')
rouge = evaluate.load('rouge')

# BLEU acepta varias referencias por predicción (lista de listas); ROUGE, una por predicción
bleu_t5 = bleu.compute(predictions=[text_t5], references=[[reference_summary]])
rouge_t5 = rouge.compute(predictions=[text_t5], references=[reference_summary])

print(f'Referencia:  {reference_summary}')
print(f'Salida T5:   {text_t5}')
print(f'BLEU:        {bleu_t5["score"]:.2f}')
print(f'ROUGE-1:     {rouge_t5["rouge1"]:.4f}')
print(f'ROUGE-L:     {rouge_t5["rougeL"]:.4f}')

> **Interpretación de la salida:**
>
> - **BLEU** (0–100) mide la **precisión** de n-gramas (hasta 4-gramas) de la predicción frente a la referencia, con una penalización si la predicción es más corta que la referencia. 0 = ningún 4-grama en común; 100 = coincidencia exacta. En oraciones cortas es muy sensible a la redacción exacta.
> - **ROUGE-1** mide la coincidencia de palabras sueltas y **ROUGE-L** la subsecuencia común más larga, que capta parte de la estructura de la oración. ROUGE se pensó para resúmenes y suele reportarse como F1 (combina precisión y *recall*).
> - Limitación común: ninguna mide equivalencia de significado. Una paráfrasis perfecta con otras palabras puede obtener un puntaje bajo, y un texto con un dato falso pero con las mismas palabras, uno alto. Métricas como **BERTScore** usan *embeddings* para acercarse mejor al significado (ver `metrics_llm_business.ipynb`).

---

## 6. Perplejidad (GPT-2)

La **perplejidad** mide qué tan bien predice un modelo de lenguaje un texto. Menor perplejidad = el modelo encuentra el texto más predecible.

$$\text{PPL}(texto) = \exp\left(-\frac{1}{N} \sum_i \log P(token_i \mid contexto)\right)$$

In [ ]:
def compute_perplexity(model, tokenizer, text: str) -> float:
    """Calcula la perplejidad de un texto bajo un modelo de lenguaje causal.

    Args:
        model: modelo de lenguaje causal (por ejemplo, GPT-2) en modo evaluación.
        tokenizer: su tokenizador.
        text: texto a evaluar.

    Returns:
        Perplejidad (menor = el modelo predice mejor el texto).
    """
    encodings = tokenizer(text, return_tensors='pt')
    input_ids = encodings.input_ids
    target_ids = input_ids.clone()   # el modelo desplaza las etiquetas internamente

    with torch.no_grad():
        outputs = model(input_ids, labels=target_ids)
        neg_log_likelihood = outputs.loss   # promedio de -log P por token

    return math.exp(neg_log_likelihood.item())


# Se reutiliza el GPT-2 ya cargado en la sección 1
ppl = compute_perplexity(gpt2_model, gpt2_tokenizer, input_text)
print(f'Perplejidad de GPT-2 sobre el texto de entrada: {ppl:.2f}')

> **Interpretación de la salida:** la perplejidad refleja cuán “sorprendido” queda el modelo con el texto. Una oración en inglés bien escrita sobre un lugar famoso (frecuente en los datos de entrenamiento de GPT-2) debería dar una perplejidad relativamente baja. Una perplejidad alta indica redacción inusual, vocabulario de un dominio especializado o un idioma con el que el modelo apenas se entrenó (prueba con una oración en español). Solo es comparable entre modelos con el **mismo tokenizador**.

## Resumen de métricas

| Métrica | Qué mide | Rango | Limitación |
|--------|----------|-------|------------|
| **BLEU** | Precisión de n-gramas frente a la referencia | 0–100 | Coincidencia exacta; no capta significado |
| **ROUGE-1** | Coincidencia de palabras sueltas | 0–1 | Solo solapamiento de palabras |
| **ROUGE-L** | Subsecuencia común más larga | 0–1 | Solo similitud estructural |
| **Perplejidad** | Confianza del modelo de lenguaje en el texto | ≥ 1 (menor = mejor) | Intrínseca; no mide utilidad para la tarea |